# ICU LOS Classification Demo

This notebook demonstrates the saved ICU length-of-stay classifier. It uses a small synthetic dataset that matches the model feature schema. The real MIMIC-IV-derived training data are restricted and are not included in this submission.

In [1]:
from pathlib import Path
import contextlib
import io
import os
import warnings

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")
warnings.filterwarnings("ignore", category=UserWarning, module="joblib.*")
warnings.filterwarnings("ignore", category=UserWarning, module="joblib.externals.loky.*")
warnings.filterwarnings("ignore", message="Could not find the number of physical cores.*")

import joblib
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, f1_score

cwd = Path.cwd()
candidate_roots = []
if cwd.name == "submission":
    candidate_roots.append(cwd)
if cwd.name == "notebooks":
    candidate_roots.append(cwd.parent)
candidate_roots.extend([cwd / "submission", cwd.parent / "submission", cwd])

ROOT = None
for candidate in candidate_roots:
    if (candidate / "models" / "icu_los_classifier.joblib").exists() and (candidate / "data" / "sample" / "icu_los_classification_sample.csv").exists():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("Could not find the submission model and sample data files.")

MODEL_PATH = ROOT / "models" / "icu_los_classifier.joblib"
SAMPLE_PATH = ROOT / "data" / "sample" / "icu_los_classification_sample.csv"

stderr_buffer = io.StringIO()
with contextlib.redirect_stderr(stderr_buffer):
    artifact = joblib.load(MODEL_PATH)
model = artifact["model"]
feature_columns = artifact["feature_columns"]
target_column = artifact["target_column"]
target_labels = artifact["target_labels"]

sample = pd.read_csv(SAMPLE_PATH)
X = sample[feature_columns]
y = sample[target_column]

with contextlib.redirect_stderr(stderr_buffer):
    pred = model.predict(X)
    proba = model.predict_proba(X)

print(f"Loaded {len(sample)} synthetic demo rows")
print(f"Model expects {len(feature_columns)} features")
print("Target labels:", target_labels)

Loaded 24 synthetic demo rows
Model expects 380 features
Target labels: {0: 'LOS < 2 days', 1: '2 <= LOS <= 7 days', 2: 'LOS > 7 days'}


In [2]:
results = sample[["subject_id", "stay_id", target_column]].copy()
results["predicted_los_category"] = pred
for idx, class_label in enumerate(model.classes_):
    results[f"prob_class_{class_label}"] = proba[:, idx]

print("Macro F1:", round(f1_score(y, pred, average="macro", zero_division=0), 3))
print("Weighted F1:", round(f1_score(y, pred, average="weighted", zero_division=0), 3))
print("Balanced accuracy:", round(balanced_accuracy_score(y, pred), 3))
print("\nConfusion matrix, rows=true and columns=predicted:")
print(confusion_matrix(y, pred, labels=[0, 1, 2]))

results.head(10)

Macro F1: 0.627
Weighted F1: 0.630
Balanced accuracy: 0.653

Confusion matrix, rows=true and columns=predicted:
[[5 1 0]
 [3 5 0]
 [2 3 5]]


subject_id,stay_id,los_category,predicted_los_category,prob_class_0,prob_class_1,prob_class_2,correct
900000,700000,0,0,0.572,0.420,0.008,True
900001,700001,0,0,0.866,0.134,0.001,True
900002,700002,0,0,0.923,0.077,0.001,True
900003,700003,0,0,0.951,0.048,0.001,True
900004,700004,0,0,0.943,0.055,0.002,True
900005,700005,1,1,0.445,0.542,0.013,True
900006,700006,1,1,0.327,0.670,0.004,True
900007,700007,1,1,0.276,0.719,0.005,True
900008,700008,1,1,0.407,0.583,0.010,True
900009,700009,1,1,0.361,0.630,0.009,True


In [3]:
print(classification_report(y, pred, labels=[0, 1, 2], target_names=[target_labels[i] for i in [0, 1, 2]], zero_division=0))

                    precision    recall  f1-score   support

      LOS < 2 days       0.50      0.83      0.62         6
2 <= LOS <= 7 days       0.56      0.62      0.59         8
      LOS > 7 days       1.00      0.50      0.67        10

          accuracy                           0.62        24
         macro avg       0.69      0.65      0.63        24
      weighted avg       0.73      0.62      0.63        24


In [4]:
results.assign(correct=results[target_column].eq(results["predicted_los_category"])).sort_values("correct").head(10)

subject_id,stay_id,los_category,predicted_los_category,prob_class_0,prob_class_1,prob_class_2,correct
900023,700023,0,1,0.286,0.708,0.006,False
900021,700021,2,0,0.363,0.355,0.282,False
900020,700020,2,1,0.112,0.594,0.293,False
900019,700019,2,1,0.112,0.646,0.242,False
900018,700018,2,1,0.288,0.445,0.267,False
900017,700017,1,0,0.919,0.080,0.001,False
900016,700016,1,0,0.558,0.435,0.008,False
900015,700015,1,0,0.949,0.044,0.007,False
900022,700022,2,0,0.433,0.397,0.169,False
900014,700014,2,2,0.161,0.415,0.424,True
